# B11-G1-8 Group Model Comparison

This notebook compares the CV-selected configurations recorded in the six individual model notebooks. The input table is a snapshot transcribed from their saved GridSearchCV outputs; rerun the individual model notebooks and refresh the comparison CSV before submission if any code, data, or environment changes.

In [5]:
from pathlib import Path
import pandas as pd

comparison_path = next(
    (candidate / "results" / "metrics" / "model_comparison.csv"
     for candidate in [Path.cwd(), *Path.cwd().resolve().parents]
     if (candidate / "results" / "metrics" / "model_comparison.csv").is_file()),
    None,
)
if comparison_path is None:
    raise FileNotFoundError("Could not find results/metrics/model_comparison.csv from the current directory or its parents.")

comparison = pd.read_csv(comparison_path)
display(comparison.sort_values("cv_macro_f1", ascending=False).reset_index(drop=True))

,model,model,selected_variant,cv_accuracy,cv_macro_f1,test_accuracy,test_precision_macro,test_recall_macro,test_macro_f1,source_notebook
0,RBF SVC,RBF SVC,MinMax scaled,0.991106,0.991127,0.979967,0.980314,0.979820,0.979877,SVC.ipynb
1,K-Nearest Neighbors,K-Nearest Neighbors,Raw pixels,0.985089,0.985166,0.977741,0.978332,0.977554,0.977734,KNearestNeighbors.ipynb
2,Random Forest,Random Forest,VarianceThreshold(threshold=0.05),0.981689,0.981762,0.972176,0.972620,0.972150,0.972205,RandomForest.ipynb
3,MLP Classifier,MLPClassifier,MinMax scaled + PCA 95%,0.972534,0.972587,0.957707,0.959144,0.957619,0.957887,MLP_PCA.ipynb
4,Logistic Regression,Logistic Regression,Before IQR outlier removal,0.971228,0.971233,0.952699,0.953776,0.952565,0.952761,LogisticRegression.ipynb
5,Decision Tree,Decision Tree,Before duplicate removal (tied with after),0.897983,0.898244,0.856427,0.858284,0.856400,0.856188,DecisionTree.ipynb


## Selection and comparison

The best overall model is selected by the highest recorded five-fold CV macro F1, not by the held-out test result. Test accuracy and test macro F1 are reported as final checks only. The CV-to-test difference below is a generalization gap, not a stability estimate; fold-level standard deviations were not saved.

In [6]:
comparison["cv_to_test_macro_f1_gap"] = (
    comparison["cv_macro_f1"] - comparison["test_macro_f1"]
)

best_cv = comparison.loc[comparison["cv_macro_f1"].idxmax()]
best_test_accuracy = comparison.loc[comparison["test_accuracy"].idxmax()]
best_test_f1 = comparison.loc[comparison["test_macro_f1"].idxmax()]

print(
    f"Best by CV macro F1: "
    f"{best_cv['it_number']} {best_cv['model']} "
    f"({best_cv['cv_macro_f1']:.4f})"
)

print(
    f"Best test accuracy: "
    f"{best_test_accuracy['it_number']} {best_test_accuracy['model']} "
    f"({best_test_accuracy['test_accuracy']:.4f})"
)

print(
    f"Best test macro F1: "
    f"{best_test_f1['it_number']} {best_test_f1['model']} "
    f"({best_test_f1['test_macro_f1']:.4f})"
)

display(
    comparison
    .sort_values("cv_macro_f1", ascending=False)
    [[
        "it_number",
        "model",
        "selected_variant",
        "test_precision_macro",
        "test_recall_macro",
        "cv_to_test_macro_f1_gap"
    ]]
    .reset_index(drop=True)
)


Best by CV macro F1:  RBF SVC (0.9911)
Best test accuracy:  RBF SVC (0.9800)
Best test macro F1:  RBF SVC (0.9799)


,model,model,selected_variant,test_precision_macro,test_recall_macro,cv_to_test_macro_f1_gap
0,RBF SVC,RBF SVC,MinMax scaled,0.980314,0.979820,0.011250
1,K-Nearest Neighbors,K-Nearest Neighbors,Raw pixels,0.978332,0.977554,0.007432
2,Random Forest,Random Forest,VarianceThreshold(threshold=0.05),0.972620,0.972150,0.009557
3,MLP Classifier,MLPClassifier,MinMax scaled + PCA 95%,0.959144,0.957619,0.014700
4,Logistic Regression,Logistic Regression,Before IQR outlier removal,0.953776,0.952565,0.018472
5,Decision Tree,Decision Tree,Before duplicate removal (tied with after),0.858284,0.856400,0.042056


## Interpretation

- The RBF SVC with MinMax scaling is the strongest recorded CV candidate and also has the highest recorded test accuracy and macro F1. Its CV-to-test difference is descriptive only; fold-to-fold variation is unavailable.
- KNN is distance-based and therefore benefits from scale-aware experiments. Its CV-selected raw-pixel variant has nearly the same CV score as the scaled variant, while the scaled variant has a higher test score; selection remains based on CV, not test.
- Random Forest is a nonlinear tree ensemble that does not require scaling. The recorded variance selection changed its CV score only slightly and did not change test accuracy.
- Logistic Regression is the linear baseline used for the outlier-removal comparison. The MLP uses nonlinear decision boundaries; PCA improved its recorded scores in the tested configuration, but a single comparison does not establish a general PCA benefit.
- The Decision Tree is the most interpretable model but has the lowest recorded scores. Duplicate removal produced identical recorded results because no duplicate training rows were removed.

Each student tests a model with a distinct model family and a student-specific preprocessing comparison. The experiments share the official test split and accuracy/macro-F1 definitions, but preprocessing differences mean this is a descriptive comparison rather than a controlled algorithm-only benchmark.

## Evaluation evidence gaps

All six model notebooks now report accuracy, macro precision, macro recall, and macro F1 for each tested variant on the untouched test split, and include a classification report and confusion matrix for the CV-selected variant. CV standard deviations and training times were not recorded, so model stability and computational performance cannot be ranked from this evidence. The comparison table contains the consistently reported aggregate metrics only.